In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import io
import requests
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

In [ ]:
# --- 1. Load Data (Robust Method) ---

file_path = 'SMSSpamCollection'
zip_name = 'smsspamcollection.zip'
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

if os.path.exists(file_path):
    print(f"Found dataset locally at {file_path}. Loading...")
    df = pd.read_csv(file_path, sep="\t", header=None, names=["label", "message"])
else:
    print(f"Dataset not found locally. Downloading from {url}...")
    try:
        response = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(response.content))
        z.extractall(".")
        print("Download and extraction complete.")
        df = pd.read_csv(file_path, sep="\t", header=None, names=["label", "message"])
    except Exception as e:
        print(f"Error downloading data: {e}")
        raise e

print("Data loaded successfully.")
print(f"Shape: {df.shape}")
display(df.head())

In [ ]:
# --- 2. Preprocessing ---

# Features and target
X = df["message"].astype(str)
y = df["label"].map({"ham": 0, "spam": 1})

# Train-test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training samples: {len(X_tr)}")
print(f"Testing samples: {len(X_te)}")

In [ ]:
# --- 3. Model: Naive Bayes ---

print("Training Naive Bayes...")
nb_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", MultinomialNB())
])

nb_model.fit(X_tr, y_tr)
y_pred_nb = nb_model.predict(X_te)

print("=== Naive Bayes (Spam Classification) ===")
print("Accuracy:", accuracy_score(y_te, y_pred_nb))
print("\nClassification Report:\n",
      classification_report(y_te, y_pred_nb, target_names=["Non-Spam", "Spam"]))
print("Confusion Matrix:\n", confusion_matrix(y_te, y_pred_nb))

In [ ]:
# --- 4. Model: Decision Tree ---

print("Training Decision Tree...")
dt_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", DecisionTreeClassifier(
        criterion="entropy",
        max_depth=25,
        min_samples_leaf=5,
        random_state=42
    ))
])

dt_model.fit(X_tr, y_tr)
dt_pred = dt_model.predict(X_te)

print("=== Decision Tree (Entropy) — Spam Classification ===")
print("Accuracy:", accuracy_score(y_te, dt_pred))
print("\nClassification Report:\n", classification_report(
    y_te, dt_pred, target_names=["Non-Spam", "Spam"], zero_division=0
))
print("Confusion Matrix:\n", confusion_matrix(y_te, dt_pred))

In [ ]:
# --- 5. Model: ANN (MLP) ---

print("Training MLP Classifier...")
ann_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("svd", TruncatedSVD(n_components=200, random_state=42)),
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        max_iter=50,  # Increased slightly for better convergence
        random_state=42
    ))
])

ann_model.fit(X_tr, y_tr)
ann_pred = ann_model.predict(X_te)

print("=== ANN (MLP) — Spam Classification ===")
print("Accuracy:", accuracy_score(y_te, ann_pred))
print("\nClassification Report:\n", classification_report(
    y_te, ann_pred, target_names=["Non-Spam", "Spam"], zero_division=0
))
print("Confusion Matrix:\n", confusion_matrix(y_te, ann_pred))

In [ ]:
# --- 6. Sample Prediction ---

sample_msg = "Congratulations! You have won a free prize. Call now!"
print("\nSample Message:", sample_msg)

pred_nb_sample = nb_model.predict([sample_msg])[0]
print("Naive Bayes Prediction:", "Spam" if pred_nb_sample == 1 else "Non-Spam")

pred_dt_sample = dt_model.predict([sample_msg])[0]
print("Decision Tree Prediction:", "Spam" if pred_dt_sample == 1 else "Non-Spam")

pred_ann_sample = ann_model.predict([sample_msg])[0]
print("MLP Prediction:", "Spam" if pred_ann_sample == 1 else "Non-Spam")